In [2]:
# Network-Level Functional Connectivity Analysis in Autism Spectrum Disorder: A Resting-State fMRI Study.

In [2]:
import nilearn
import warnings
warnings.filterwarnings("ignore")

In [3]:
from nilearn.datasets import fetch_abide_pcp
# Fetch the full data and update phenotypic data and cross_validation
abide = fetch_abide_pcp(derivatives = ['func_preproc'], pipeline = 'cpac', quality_checked = True)

[get_dataset_dir] Dataset found in C:\Users\mutlu\nilearn_data\ABIDE_pcp

In [4]:
from nilearn.datasets import fetch_abide_pcp
import pandas as pd

# Derivatives = preprocessed data
# Pipeline = CPAC (motion correction, slice timing, normalization to MNI space, nuisance regression)
# Quality Checked = True (only good quality scans included)
abide = fetch_abide_pcp(derivatives=['func_preproc'], pipeline='cpac', quality_checked=True)

func_imgs = abide.func_preproc
df = pd.DataFrame(abide.phenotypic).reset_index(drop=True)

#ensuring the numbers of ASD and control
asd_indices = df.index[df["DX_GROUP"] == 1].tolist()
control_indices = df.index[df["DX_GROUP"] == 2].tolist()

print(f"ASD subjects: {len(asd_indices)}")
print(f"Control subjects: {len(control_indices)}")

[get_dataset_dir] Dataset found in C:\Users\mutlu\nilearn_data\ABIDE_pcp

ASD subjects: 403
Control subjects: 468


In [5]:
from nilearn import datasets

atlas = datasets.fetch_atlas_schaefer_2018(n_rois=200, yeo_networks=7, resolution_mm=2)
labels = [lbl.decode("utf-8") for lbl in atlas["labels"]]

# DMN regions are the ones labeled "Default" in the Schaefer/Yeo 7-network scheme
dmn_indices = [i for i, name in enumerate(labels) if "Default" in name]
dmn_labels = [labels[i] for i in dmn_indices]

# The DMN labelled regions from the atlas are the regions I analyzed
print(f"Number of DMN regions: {len(dmn_indices)}")
for dmn_label in dmn_labels:
    print(dmn_label)

[get_dataset_dir] Dataset found in C:\Users\mutlu\nilearn_data\schaefer_2018

Number of DMN regions: 46
7Networks_LH_Default_Temp_1
7Networks_LH_Default_Temp_2
7Networks_LH_Default_Temp_3
7Networks_LH_Default_Temp_4
7Networks_LH_Default_Temp_5
7Networks_LH_Default_Par_1
7Networks_LH_Default_Par_2
7Networks_LH_Default_Par_3
7Networks_LH_Default_Par_4
7Networks_LH_Default_PFC_1
7Networks_LH_Default_PFC_2
7Networks_LH_Default_PFC_3
7Networks_LH_Default_PFC_4
7Networks_LH_Default_PFC_5
7Networks_LH_Default_PFC_6
7Networks_LH_Default_PFC_7
7Networks_LH_Default_PFC_8
7Networks_LH_Default_PFC_9
7Networks_LH_Default_PFC_10
7Networks_LH_Default_PFC_11
7Networks_LH_Default_PFC_12
7Networks_LH_Default_PFC_13
7Networks_LH_Default_pCunPCC_1
7Networks_LH_Default_pCunPCC_2
7Networks_LH_Default_pCunPCC_3
7Networks_LH_Default_pCunPCC_4
7Networks_LH_Default_PHC_1
7Networks_RH_Default_Par_1
7Networks_RH_Default_Par_2
7Networks_RH_Default_Par_3
7Networks_RH_Default_Temp_1
7Networks_RH_Default_Temp_2
7Networks_RH_Default_Temp_3
7Networks_RH_Default_Temp_4
7Networks_RH_Default_Temp_5

In [6]:
import os
import numpy as np

# --------------------------------------------------------------------
# STEP 1: Full-brain extraction (200 regions)
# --------------------------------------------------------------------
# Each subject's preprocessed fMRI scan was passed through a NiftiLabelsMasker
# built on the Schaefer 2018 atlas (200 ROIs, 7 Yeo networks). This produced
# one (timepoints x 200) time-series array per subject, saved to
# TS_200ROI_ASD_simple/ and TS_200ROI_CONTROL_simple/. Loading those results
# here rather than re-running extraction.

def load_timeseries(folder):
    files = sorted(os.listdir(folder), key=lambda x: int(x.split('_')[1].split('.')[0]))
    return [np.load(os.path.join(folder, f)) for f in files], files

asd_200roi, asd_files = load_timeseries("TS_200ROI_ASD_simple")
control_200roi, control_files = load_timeseries("TS_200ROI_CONTROL_simple")

print(f"ASD subjects (200-ROI): {len(asd_200roi)}")
print(f"Control subjects (200-ROI): {len(control_200roi)}")
print(f"Example shape: {asd_200roi[0].shape}" + " for this example subjects, (timepoint, 200) since 200 regions are extracted")  # (timepoints, 200)


# --------------------------------------------------------------------
# STEP 2: DMN subsetting (200 -> 46 regions)
# --------------------------------------------------------------------
# From the 200 whole-brain regions, only the 46 labeled "Default" (the
# Default Mode Network, per the Yeo 7-network scheme) were kept. This
# subsetting was already applied and saved to DMN_ASD/ and DMN_CONTROL/.
# Loading those directly here.

asd_dmn, _ = load_timeseries("DMN_ASD")
control_dmn, _ = load_timeseries("DMN_CONTROL")

print(f"\nASD subjects (DMN, 46 regions): {len(asd_dmn)}")
print(f"Control subjects (DMN, 46 regions): {len(control_dmn)}")
print(f"Example shape: {asd_dmn[0].shape}" + " for this example subjects, (timepoint, 46) since 46 DMN regions are extracted only")  # (timepoints, 46)

ASD subjects (200-ROI): 403
Control subjects (200-ROI): 468
Example shape: (196, 200) for this example subjects, (timepoint, 200) since 200 regions are extracted

ASD subjects (DMN, 46 regions): 403
Control subjects (DMN, 46 regions): 468
Example shape: (196, 46) for this example subjects, (timepoint, 46) since 46 DMN regions are extracted only


In [7]:
from nilearn.connectome import ConnectivityMeasure

# --------------------------------------------------------------------
# How the connectivity vectors were built (already run, shown for reference)
# --------------------------------------------------------------------
# Each subject's (timepoints, 46) DMN time-series was converted into a
# 46 x 46 Pearson correlation matrix, capturing functional connectivity
# between every pair of DMN regions. Since timepoints vary per subject,
# this collapses that variable dimension into a fixed-size representation.
#
# The matrix is symmetric and the diagonal (self-correlation = 1) is
# uninformative, so only the unique upper-triangle values were kept
# (vectorize=True, discard_diagonal=True), giving 46 x 45 / 2 = 1035
# edges per subject.
#
# correlation = ConnectivityMeasure(
#     kind='correlation',
#     vectorize=True,
#     discard_diagonal=True,
#     standardize=True
# )
#
# for subject_ts in dmn_time_series:
#     vec = correlation.fit_transform([subject_ts])[0]  # shape: (1035,)

# --------------------------------------------------------------------
# Loading the already-built vectors
# --------------------------------------------------------------------
ASD_vectors = np.load("ASD_DMN_vectors.npy")
CONTROL_vectors = np.load("CONTROL_DMN_vectors.npy")

print(f"ASD_vectors shape: {ASD_vectors.shape}")         # (403, 1035)
print(f"CONTROL_vectors shape: {CONTROL_vectors.shape}")  # (468, 1035)

ASD_vectors shape: (403, 1035)
CONTROL_vectors shape: (468, 1035)


In [8]:
# import numpy as np

# --------------------------------------------------------------------
# Fisher's Z transformation
# --------------------------------------------------------------------
# Pearson correlations are bounded (-1, 1), which distorts their distribution
# and violates the normality assumption needed for an accurate t-test.
# Fisher's Z (arctanh) normalizes the values before statistical comparison.
#
# Source: Chen, S., Xing, Y., & Kang, J. (2017). Latent and abnormal
# functional connectivity circuits in autism spectrum disorder.
# Frontiers in Neuroscience, 11, 125.

ASD_z = np.arctanh(ASD_vectors)
CONTROL_z = np.arctanh(CONTROL_vectors)

print(f"ASD_z shape: {ASD_z.shape}")         # (403, 1035)
print(f"CONTROL_z shape: {CONTROL_z.shape}")  # (468, 1035)

ASD_z shape: (403, 1035)
CONTROL_z shape: (468, 1035)


In [9]:
import numpy as np
from scipy.stats import ttest_ind
from itertools import combinations

# Assumes ASD_z, CONTROL_z already exist from the Fisher's z step,
# and dmn_labels (46 DMN region names) already exists from the atlas cell

# --- t-test across all 1035 DMN edges ---
# Welch T-test since the varbales are not the same size, performed by
# equal_var = False
t_vals, p_vals = ttest_ind(ASD_z, CONTROL_z, equal_var=False)

# --- Uncorrected masks (raw p < 0.05) ---
# No extra FDR connection is performed for this file
hyper_mask_uncorrected = (t_vals > 0) & (p_vals < 0.05)
hypo_mask_uncorrected  = (t_vals < 0) & (p_vals < 0.05)

print(f"[Uncorrected] Hyper: {hyper_mask_uncorrected.sum()}, Hypo: {hypo_mask_uncorrected.sum()}, "
      f"Total: {hyper_mask_uncorrected.sum() + hypo_mask_uncorrected.sum()} out of {len(p_vals)}")

# --- edge index -> region pair name lookup ---
# NOTE: assumes the 1035-length vectors are the upper triangle of the 46x46
# matrix in dmn_labels order (itertools.combinations order).
edge_labels = list(combinations(dmn_labels, 2))

# --- top 5 hyper edges (uncorrected) ---
hyper_idx = np.where(hyper_mask_uncorrected)[0]
top5_hyper = hyper_idx[np.argsort(t_vals[hyper_idx])[-5:]]
print("\nTop 5 hyper edges (uncorrected):")
for idx in top5_hyper:
    r1, r2 = edge_labels[idx]
    print(f"Edge {idx} ({r1} -- {r2}): t = {t_vals[idx]:.3f}, p = {p_vals[idx]:.5f}")

# --- top 5 hypo edges (uncorrected) ---
hypo_idx = np.where(hypo_mask_uncorrected)[0]
top5_hypo = hypo_idx[np.argsort(t_vals[hypo_idx])[:5]]
print("\nTop 5 hypo edges (uncorrected):")
for idx in top5_hypo:
    r1, r2 = edge_labels[idx]
    print(f"Edge {idx} ({r1} -- {r2}): t = {t_vals[idx]:.3f}, p = {p_vals[idx]:.5f}")

[Uncorrected] Hyper: 0, Hypo: 481, Total: 481 out of 1035

Top 5 hyper edges (uncorrected):

Top 5 hypo edges (uncorrected):
Edge 960 (7Networks_RH_Default_Temp_4 -- 7Networks_RH_Default_PFCdPFCm_2): t = -6.888, p = 0.00000
Edge 460 (7Networks_LH_Default_PFC_3 -- 7Networks_RH_Default_Temp_3): t = -6.831, p = 0.00000
Edge 490 (7Networks_LH_Default_PFC_4 -- 7Networks_RH_Default_Par_3): t = -6.679, p = 0.00000
Edge 477 (7Networks_LH_Default_PFC_4 -- 7Networks_LH_Default_PFC_8): t = -6.522, p = 0.00000
Edge 450 (7Networks_LH_Default_PFC_3 -- 7Networks_LH_Default_pCunPCC_1): t = -6.484, p = 0.00000


In [10]:
import numpy as np
from scipy.stats import ttest_ind, loguniform
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

ASD = np.load("ASD_DMN_vectors.npy")
CONTROL = np.load("CONTROL_DMN_vectors.npy")

ASD_z = np.arctanh(np.clip(ASD, -0.999999, 0.999999))
CONTROL_z = np.arctanh(np.clip(CONTROL, -0.999999, 0.999999))

X_all = np.vstack((ASD_z, CONTROL_z))
y = np.array([1]*ASD_z.shape[0] + [0]*CONTROL_z.shape[0])

seeds = [0, 1, 42, 123, 2024]

param_distributions = [
    {"svc__kernel": ["linear"], "svc__C": loguniform(1e-5, 1e2),
     "svc__class_weight": [None, "balanced"]},
    {"svc__kernel": ["rbf"], "svc__C": loguniform(1e-5, 1e2),
     "svc__gamma": loguniform(1e-5, 1e1), "svc__class_weight": [None, "balanced"]}
]

accuracies = []
f1_scores = []
n_features_list = []

for s in seeds:
    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y, test_size=0.2, random_state=s, stratify=y
    )

    t_vals, p_vals = ttest_ind(X_train[y_train == 1], X_train[y_train == 0], equal_var = False)
    mask_uncorrected = (t_vals < 0) & (p_vals < 0.05)

    X_train_sel, X_test_sel = X_train[:, mask_uncorrected], X_test[:, mask_uncorrected]
    n_features_list.append(mask_uncorrected.sum())

    pipe = Pipeline([("scaler", StandardScaler()), ("svc", SVC())])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=s)

    search = RandomizedSearchCV(
        pipe, param_distributions, n_iter=100, scoring="f1_weighted",
        cv=cv, n_jobs=-1, random_state=s, verbose=0
    )
    search.fit(X_train_sel, y_train)
    y_pred = search.best_estimator_.predict(X_test_sel)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    accuracies.append(acc)
    f1_scores.append(f1)

    print(f"Seed {s}: {mask_uncorrected.sum()} features | "
          f"Accuracy: {acc:.3f} | F1: {f1:.3f} | "
          f"Best params: {search.best_params_}")

print(f"\nMean accuracy: {np.mean(accuracies):.3f} ± {np.std(accuracies):.3f}")
print(f"Mean F1: {np.mean(f1_scores):.3f} ± {np.std(f1_scores):.3f}")
print(f"Feature count range across seeds: {min(n_features_list)}–{max(n_features_list)}")

Seed 0: 581 features | Accuracy: 0.577 | F1: 0.577 | Best params: {'svc__C': 0.0004122838736517057, 'svc__class_weight': 'balanced', 'svc__kernel': 'linear'}
Seed 1: 564 features | Accuracy: 0.606 | F1: 0.595 | Best params: {'svc__C': 0.0004902572075068691, 'svc__class_weight': None, 'svc__kernel': 'linear'}
Seed 42: 384 features | Accuracy: 0.617 | F1: 0.617 | Best params: {'svc__C': 0.001433591914748391, 'svc__class_weight': 'balanced', 'svc__kernel': 'linear'}
Seed 123: 350 features | Accuracy: 0.629 | F1: 0.607 | Best params: {'svc__C': 0.0004929391727469952, 'svc__class_weight': None, 'svc__kernel': 'linear'}
Seed 2024: 523 features | Accuracy: 0.674 | F1: 0.661 | Best params: {'svc__C': 51.29122959844051, 'svc__class_weight': None, 'svc__gamma': 1.2558206176931213e-05, 'svc__kernel': 'rbf'}

Mean accuracy: 0.621 ± 0.032
Mean F1: 0.611 ± 0.028
Feature count range across seeds: 350–581
